In [10]:
from google_play_scraper import reviews, Sort
import pandas as pd
from datetime import datetime
import time
import os

### WEB SCRAPING 

In [23]:
# App IDs for Ethiopian banks
# Note: Verify these IDs on Google Play Store
BANK_APPS = {
    'CBE': 'com.combanketh.mobilebanking',
    'BOA': 'com.boa.boaMobileBanking',
    'Dashen Bank': 'com.dashen.dashensuperapp'
}

print("Bank App IDs:")
for bank, app_id in BANK_APPS.items():
    print(f"  {bank}: {app_id}")

Bank App IDs:
  CBE: com.combanketh.mobilebanking
  BOA: com.boa.boaMobileBanking
  Dashen Bank: com.dashen.dashensuperapp


Test each bank with just 5 reviews to verify IDs work

In [34]:
print("Testing app IDs...")
print("="*50)

working_banks = {}
broken_banks = []

for bank_name, app_id in BANK_APPS.items():
    try:
        result, _ = reviews(app_id, count=5)
        if len(result) > 0:
            print(f"{bank_name}: Working! Found {len(result)} reviews")
            working_banks[bank_name] = app_id
        else:
            print(f" {bank_name}: App exists but no reviews")
            broken_banks.append(bank_name)
    except Exception as e:
        print(f"{bank_name}: Error - {str(e)[:50]}")
        broken_banks.append(bank_name)

print("\n" + "="*50)
print(f"Working banks: {list(working_banks.keys())}")
if broken_banks:
    print(f"Banks with issues: {broken_banks}")
    print("You may need to find correct app IDs for these banks")

Testing app IDs...
CBE: Working! Found 5 reviews
BOA: Working! Found 5 reviews
Dashen Bank: Working! Found 5 reviews

Working banks: ['CBE', 'BOA', 'Dashen Bank']


In [25]:
def scrape_bank_reviews(app_id, bank_name, target_count=450):
    """Scrape reviews INCLUDING unique review_id"""
    print(f"\n📱 Scraping {bank_name}...")
    
    try:
        result, _ = reviews(
            app_id,
            lang='en',
            country='et',
            sort=Sort.NEWEST,
            count=target_count
        )
        
        reviews_data = []
        for review in result:
            reviews_data.append({
                'review_id': review['reviewId'],
                'review': review['content'],
                'rating': review['score'],
                'date': review['at'].strftime('%Y-%m-%d'),
                'bank': bank_name,
                'source': 'Google Play'
            })
        
        return pd.DataFrame(reviews_data)
        
    except Exception as e:
        print(f" Error: {e}")
        return pd.DataFrame()

In [26]:
# Scrape all three banks
all_reviews = []

print("="*60)
print("STARTING FULL SCRAPE FOR ALL BANKS")
print("="*60)

for bank_name, app_id in BANK_APPS.items():
    # Scrape the bank
    df = scrape_bank_reviews(app_id, bank_name, target_count=450)
    
    if not df.empty:
        all_reviews.append(df)
        print(f"  Added {len(df)} reviews from {bank_name}")
    else:
        print(f" Failed to get reviews from {bank_name}")
    
    # Wait 3 seconds between requests to be respectful
    if bank_name != list(BANK_APPS.keys())[-1]:  # Don't wait after last one
        print(" Waiting 3 seconds before next bank...")
        time.sleep(3)

print("\n" + "="*60)
print(f"Total banks scraped: {len(all_reviews)}")
print(f"Total raw reviews collected: {sum(len(df) for df in all_reviews)}")
print("="*60)

STARTING FULL SCRAPE FOR ALL BANKS

📱 Scraping CBE...
  Added 450 reviews from CBE
 Waiting 3 seconds before next bank...

📱 Scraping BOA...
  Added 450 reviews from BOA
 Waiting 3 seconds before next bank...

📱 Scraping Dashen Bank...
  Added 450 reviews from Dashen Bank

Total banks scraped: 3
Total raw reviews collected: 1350


In [39]:
# Combine all DataFrames
if all_reviews:
    combined_df = pd.concat(all_reviews, ignore_index=True)
    print(f" Combined all reviews: {len(combined_df)} total")
    print(f"\nReviews per bank:")
    print(combined_df['bank'].value_counts())
else:
    print("No reviews collected from any bank")
    combined_df = pd.DataFrame()


 Combined all reviews: 1350 total

Reviews per bank:
bank
CBE            450
BOA            450
Dashen Bank    450
Name: count, dtype: int64


In [50]:
# In a Jupyter cell, run
print(os.getcwd())  # Current directory
print(os.listdir('.'))  # Files in current directory
print(os.listdir('notebooks') if os.path.exists('notebooks') else "notebooks folder doesn't exist")

c:\Users\sisay\OneDrive\Documents\kaim\fintech-review-analytics\notebooks
['README.md', 'web_scraping.ipynb', '__init__.py']
notebooks folder doesn't exist
